# Optimization Campaign (HITL)

Interactive notebook for running prompt optimization campaigns with full human-in-the-loop control.

**Workflow:** Config → Load Data → Diagnostics → Baseline Eval → Optimization Round → LLM Suggestions → Repeat

**Prerequisites:**
1. **TermNorm backend running** at `http://127.0.0.1:8000` — required for the first run to sync experiment data with traces. Once synced, the data is cached locally and the backend is no longer needed.
2. **Groq API key** set in `.env`.
3. After the first sync, **restart the kernel** once so the setup cell loads the freshly cached data.

## 1. Setup & Load Data

On first run (or after clearing cache), you'll see the auto-sync message — this is expected:
```
Cached data has no traces — syncing from http://127.0.0.1:8000 ...
Experiment : production_historical
Mappings   : 887 total, 812 with verified ground truth
Queries    : 40  |  Session terms: 93
Loaded 40 eval queries
Ready.
```

In [ ]:
#@title Setup & imports
%load_ext autoreload
%autoreload 2

import json
import os

import pandas as pd

from _campaign_lib import (
    init_services, analyze_candidate_coverage,
    load_baseline_prompt, evaluate_prompt,
    generate_candidates, select_round_winner, generate_suggestions,
    display_suggestions, save_campaign_winner,
    # Grid search
    DEFAULT_GRID_AXES, restructure_context, validate_grid_config,
    build_grid_combinations, run_grid_search, display_grid_results,
    select_grid_winner, analyze_grid_results, load_eval_dataset,
)

svc = await init_services()

eval_data = load_eval_dataset(
    svc["store"], svc["backend_id"], svc["experiment_id"],
)
if not eval_data:
    raise RuntimeError(
        "No evaluation data in project store. "
        "Generate data first (e.g. run termnorm_backend.ipynb or another data source)."
    )

campaign_rounds = []
print("Ready.")

## 2. Campaign Config

Edit this cell and re-run to change settings for pipeline parameters (read-only reference), optimization, and the evaluation LLM. After each round, the LLM suggestion cell will print a modified config you can copy back here.

In [ ]:
campaign_config = {
    "pipeline_params": {                 # Read-only reference (used by termnorm_backend.ipynb replay)
        "max_sites": 7,                  # Web: pages fetched
        "num_results": 20,               # Web: search results count
        "content_char_limit": 800,       # Web: chars per page
        "raw_content_limit": 5000,       # LLM1: research text input
        "profiling_temperature": 0.3,    # LLM1: temperature
        "profiling_max_tokens": 1800,    # LLM1: output limit
        "ranking_temperature": 0,        # LLM2: temperature
        "ranking_max_tokens": 4000,      # LLM2: output limit
        "ranking_sample_size": 20,       # LLM2: candidates to rerank
        "max_token_candidates": 20,      # Token matching: kept
        "relevance_weight_core": 0.7,    # Scoring: core vs spec weight
    },
    "optimization": {
        "n_variants": 5,
        "creativity": 0.7,
        "improvement_threshold": 0.01,
        "max_rounds": 3,
    },
    "eval_llm": {
        "model": "meta-llama/llama-4-maverick-17b-128e-instruct",
        "provider_url": "https://api.groq.com/openai/v1/chat/completions",
        "temperature": 0,
        "max_tokens": 4000,
    },
    "grid_search": {
        "context": "A terminology normalization pipeline that matches raw material "
                   "descriptions to standardized database terms using entity profiling "
                   "and candidate ranking.",
        # OR provide structured fields directly:
        # "context_fields": {"persona": "...", "task_intent": "...", ...},
        "query_limit": 35,           # queries per combo (recommended baseline)
        "max_combinations": 0,       # 0=full grid, N=random sample
        "seed": 42,
        "top_k": 5,
        "use_defaults": True,        # use DEFAULT_GRID_AXES library
        # "custom_axes": {...},      # override specific axes
    },
}

print(json.dumps(campaign_config, indent=2))

## 3. Diagnostic — Candidate Coverage

For each query: is the ground truth in the token-matched candidates? At what rank? This determines whether reranker optimization is viable (ground truth must be in the candidate set for the reranker to promote it).

In [ ]:
#@title Candidate coverage analysis
cov_df = analyze_candidate_coverage(eval_data)

In [ ]:
#@title Sample entity profiles (qualitative check)
n_samples = 3
samples = [r for r in eval_data if r.get("pipeline_data", {}).get("entity_profile")][:n_samples]

for i, s in enumerate(samples):
    profile = s["pipeline_data"]["entity_profile"]
    print(f"--- Sample {i+1}: {s['query'][:60]} ---")
    print(f"  Core concept: {profile.get('core_concept', '?')}")
    print(f"  Profile keys: {list(profile.keys())}")
    print(f"  Ground truth: {s['ground_truth']}")
    candidates = s.get("pipeline_data", {}).get("token_matched_candidates", [])[:5]
    print(f"  Top 5 candidates: {[c[0] if isinstance(c, (list,tuple)) else c for c in candidates]}")
    print()

## 4. Load Baseline & Evaluate

Load the current `llm_ranking` prompt from the synced experiment, wrap it in a PromptState, and evaluate it locally using cached pipeline data.

In [ ]:
#@title Load baseline prompt
baseline = load_baseline_prompt(svc["exp_data"])
GROQ_API_KEY = os.environ.get("GROQ_API_KEY", "")
print(f"Evaluation data: {len(eval_data)} queries (loaded in Section 1)")

In [ ]:
#@title Evaluate baseline prompt
baseline_results = await evaluate_prompt(
    baseline, eval_data, campaign_config["eval_llm"], GROQ_API_KEY, label="Baseline",
)
baseline_hits = sum(1 for r in baseline_results if r["hit"])
baseline_accuracy = baseline_hits / len(baseline_results) if baseline_results else 0
campaign_rounds = [{
    "round": 0, "label": "baseline", "prompt_state": baseline,
    "accuracy": baseline_accuracy, "hits": baseline_hits,
    "total": len(baseline_results), "results": baseline_results,
}]
failures = [r for r in baseline_results if not r["hit"] and not r["error"]]
for r in failures[:5]:
    print(f"  MISS: {r['query'][:55]}  |  Pred: {r['predicted'][:35]}  |  GT: {r['ground_truth'][:35]}")

## 4.5 Grid Search — Landscape Exploration

**What:** Systematic sweep of the prompt configuration space (Layer 1 fields) using a cartesian product of default axis variations. Maps the accuracy landscape before hill-climbing.

**When to use:** First time on a new domain, or when you have little prior knowledge about which prompt components matter most. If you already have a good starting point, skip to Section 5.

**What you get:** Ranked starting points, which dimensions matter most (marginal stats), interaction effects between fields (heatmaps), and LLM-analyzed insights.

**How to read results:**
- **Ranked table** — best combos at the top; use the winner as your campaign seed
- **Marginal stats** — which axis values have the highest mean accuracy across all combos
- **Pairwise heatmaps** — green = good interaction, red = bad; look for synergies and conflicts
- **LLM analysis** — automated pattern recognition across the grid results

In [ ]:
#@title 4.5a — Restructure context into Layer 1 fields
gs = campaign_config["grid_search"]
GROQ_API_KEY = os.environ.get("GROQ_API_KEY", "")

context_input = gs.get("context_fields", gs.get("context", ""))
structured_fields = await restructure_context(context_input, campaign_config["eval_llm"], GROQ_API_KEY)
print("\nReview the restructured fields above. Edit grid_search.context_fields in Section 2 to override.")

In [ ]:
#@title 4.5b — Build grid combinations
# Load evaluation data for grid search (subset of main eval_data)
gs = campaign_config["grid_search"]
grid_eval_data = load_eval_dataset(
    svc["store"], svc["backend_id"], svc["experiment_id"],
    query_limit=gs.get("query_limit", 35),
)
if not grid_eval_data:
    raise RuntimeError(
        "No evaluation data found. Generate data first "
        "(e.g. run termnorm_backend.ipynb or another data source)."
    )

# Build grid axes
grid_axes = dict(DEFAULT_GRID_AXES) if gs.get("use_defaults", True) else {}
if gs.get("custom_axes"):
    grid_axes.update(gs["custom_axes"])

# Apply restructured fields as baseline overrides
grid_baseline = baseline.derive(**{k: v for k, v in structured_fields.items() if v}, changes_description="grid_baseline")

grid_meta = validate_grid_config(grid_axes, grid_baseline)
grid_combinations, grid_ps_lookup = build_grid_combinations(
    grid_axes, grid_baseline,
    max_combinations=gs.get("max_combinations", 0),
    seed=gs.get("seed", 42),
)
est_calls = len(grid_combinations) * len(grid_eval_data)
print(f"\nEstimated LLM calls: {est_calls} ({len(grid_combinations)} combos x {len(grid_eval_data)} queries)")

In [ ]:
#@title 4.5c — Run grid search
grid_df = await run_grid_search(
    grid_combinations, grid_ps_lookup, grid_eval_data,
    campaign_config["eval_llm"], GROQ_API_KEY,
)

In [ ]:
#@title 4.5d — Display grid results
display_grid_results(grid_df, grid_axes, top_k=gs.get("top_k", 5))

In [ ]:
#@title 4.5e — LLM analysis of grid results
grid_analysis = await analyze_grid_results(
    grid_df, grid_axes, campaign_config["eval_llm"], GROQ_API_KEY,
)

In [ ]:
#@title 4.5f — Select grid winner and seed campaign
grid_winner = select_grid_winner(grid_df, grid_ps_lookup)
campaign_rounds.append(grid_winner)

# Print Layer 1 breakdown
winner_ps = grid_winner["prompt_state"]
print(f"\nLayer 1 breakdown of grid winner:")
for field in ("persona", "task_intent", "problem_description", "instruction", "thinking_style", "answer_format"):
    val = getattr(winner_ps, field)
    if val:
        print(f"  {field}: {val[:80]}{'...' if len(val) > 80 else ''}")
    else:
        print(f"  {field}: (empty)")

print(f"\nRendered prompt preview ({len(winner_ps.render())} chars):")
print(winner_ps.render()[:500])
if len(winner_ps.render()) > 500:
    print("...")
print(f"\nGrid winner appended to campaign_rounds as round '{grid_winner['round']}'. Proceed to Section 5 for optimization.")

## 5. Run One Optimization Round

Analyze failures from the current best, generate N candidate prompts, evaluate all locally, select the round winner. Re-run this section for additional rounds.

In [ ]:
#@title Run optimization round
opt = campaign_config["optimization"]
current_best = campaign_rounds[-1]
print(f"=== ROUND {len(campaign_rounds)} === Current best: {current_best['label']} ({current_best['accuracy']:.1%})\n")

candidates = await generate_candidates(
    current_best["prompt_state"], current_best["accuracy"], current_best["results"],
    opt["n_variants"], opt["creativity"], campaign_config["eval_llm"], GROQ_API_KEY,
)
all_candidate_results = {}
for idx, c in enumerate(candidates):
    all_candidate_results[c.id] = await evaluate_prompt(
        c, eval_data, campaign_config["eval_llm"], GROQ_API_KEY, label=f"Candidate {idx+1}",
    )
round_entry = select_round_winner(candidates, all_candidate_results, current_best, opt["improvement_threshold"])
round_entry["round"] = len(campaign_rounds)
campaign_rounds.append(round_entry)

## 6. LLM Suggestion for Next Round (HITL)

After each round, the LLM analyzes failures and suggests:
1. Failure pattern analysis
2. Parameter change suggestions
3. Prompt phrase fragments to adopt
4. Suggested next `campaign_config`

**Review the suggestions, edit the config cell (Section 2), then re-run Sections 5-6.**

In [ ]:
#@title Generate LLM suggestions for next round
suggestions = await generate_suggestions(
    campaign_rounds, eval_data, campaign_config, campaign_config["eval_llm"], GROQ_API_KEY,
)
display_suggestions(suggestions, len(campaign_rounds))
print(f"\n--- SUGGESTED CONFIG (copy to Section 2) ---")
print(json.dumps(suggestions.get("suggested_config", campaign_config), indent=2))

## 7. Campaign Summary

Compare all rounds, track per-query flips, display the PromptState lineage chain, and save the winner.

In [ ]:
#@title Campaign comparison table
rows = []
for rd in campaign_rounds:
    rows.append({
        "round": rd["round"],
        "label": rd["label"][:40],
        "hit@1": rd["hits"],
        "total": rd["total"],
        "accuracy": f"{rd['accuracy']:.1%}",
        "prompt_id": rd["prompt_state"].id[:12],
    })

print(f"CAMPAIGN SUMMARY ({len(campaign_rounds)} rounds)")
print(f"{'='*70}")
display(pd.DataFrame(rows))

In [ ]:
#@title Per-query flip tracking (baseline vs final)
if len(campaign_rounds) >= 2:
    base_r = campaign_rounds[0]["results"]
    final_r = campaign_rounds[-1]["results"]

    flips = []
    for br, fr in zip(base_r, final_r):
        b_hit = br["hit"]
        f_hit = fr["hit"]
        if b_hit != f_hit:
            flips.append({
                "query": br["query"][:50],
                "flip": "MISS->HIT" if f_hit else "HIT->MISS",
                "base_pred": br["predicted"][:35],
                "final_pred": fr["predicted"][:35],
                "ground_truth": br["ground_truth"][:35],
            })

    gained = sum(1 for f in flips if f["flip"] == "MISS->HIT")
    lost = sum(1 for f in flips if f["flip"] == "HIT->MISS")

    print(f"FLIP TRACKING (baseline -> round {campaign_rounds[-1]['round']})")
    print(f"  Queries gained (MISS->HIT): {gained}")
    print(f"  Queries lost (HIT->MISS):   {lost}")
    print(f"  Net change:                 {gained - lost:+d}")
    print()
    if flips:
        display(pd.DataFrame(flips))
else:
    print("Need at least 2 rounds for flip tracking.")

In [ ]:
#@title PromptState lineage chain
print("LINEAGE CHAIN")
print("="*50)
for i, rd in enumerate(campaign_rounds):
    ps = rd["prompt_state"]
    parent = ps.parent_id[:12] if ps.parent_id else "root"
    arrow = "  " if i == 0 else "  -> "
    print(f"{arrow}[{ps.id[:12]}] Round {rd['round']}: {rd['label'][:40]} ({rd['accuracy']:.1%})")
    if ps.parent_id:
        print(f"       parent: {parent}  |  changes: {ps.changes_description or 'none'}")

In [ ]:
#@title Save winner
save_campaign_winner(campaign_rounds, campaign_config, svc["store"], svc["backend_id"])